In [1]:
import pandas as pd

In [2]:
import torch

In [3]:
from torch import nn, optim

In [4]:
import torch.nn.functional as F

In [5]:
from string import punctuation

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
df = pd.read_csv("C:\\git\\dados\\nlp\\news_sentiment_analysis.csv", encoding="utf-8")

In [17]:
dicionario_sentimentos = {}
indice_sentimentos = 0
for sentimento in df["Sentiment"]:
    if sentimento not in dicionario_sentimentos:
        dicionario_sentimentos[sentimento] = indice_sentimentos
        indice_sentimentos += 1
print(dicionario_sentimentos)

{'positive': 0, 'neutral': 1, 'negative': 2}


In [21]:
df["sentimento_number"] = [dicionario_sentimentos[s] for s in df["Sentiment"]]
df

,Description,Sentiment,sentimento_numerico,Type,TextoLimpo,sentimento_number
0,"ST. GEORGE — Kaitlyn Larson, a first-year teac...",positive,"[1, 0, 0]",Business,st george kaitlyn larson a firstyear teacher ...,0
1,"Harare, Zimbabwe – Local businesses are grappl...",neutral,"[0, 1, 0]",Business,harare zimbabwe – local businesses are grappli...,1
2,(marketscreener.com) Billionaire Elon Musk has...,positive,"[1, 0, 0]",Business,marketscreenercom billionaire elon musk has do...,0
3,(marketscreener.com) A U.S. trade regulator on...,negative,"[0, 0, 1]",Business,marketscreenercom a us trade regulator on frid...,2
4,4.5 million households in the U.S. have solar ...,positive,"[1, 0, 0]",Business,45 million households in the us have solar pan...,0
...,...,...,...,...,...,...
3495,QRG Capital Management Inc. increased its stak...,positive,"[1, 0, 0]",Technology,qrg capital management inc increased its stake...,0
3496,QRG Capital Management Inc. bought a new posit...,positive,"[1, 0, 0]",Technology,qrg capital management inc bought a new positi...,0
3497,QRG Capital Management Inc. boosted its stake ...,positive,"[1, 0, 0]",Technology,qrg capital management inc boosted its stake i...,0
3498,"WESTFORD, Mass., July 18, 2024 /PRNewswire/ --...",neutral,"[0, 1, 0]",Technology,westford mass july 18 2024 prnewswire accordin...,1


In [25]:
sentimento_numero = torch.tensor(df["sentimento_number"])

# Aplicando One-Hot Encoding
df["sentimento_one_hot"] = F.one_hot(sentimento_numero, num_classes=3).tolist()

In [9]:
# df["sentimento_numerico"] = df["Sentiment"].map( {"positive": [1, 0, 0], "negative": [0, 0, 1], "neutral": [0, 1, 0]} )

In [26]:
df = df[["Description", "Sentiment", "sentimento_one_hot", "Type"]]

In [27]:
def limpar_texto( texto : str ) -> str:
    trans_table = str.maketrans("—", " ", punctuation)
    texto_limpo = texto.lower().translate( trans_table ).replace("  ", " ")
    # texto_limpo.replace("  ", " ")
    return texto_limpo

In [28]:
df["TextoLimpo"] = df["Description"].apply( limpar_texto )
df

,Description,Sentiment,sentimento_one_hot,Type,TextoLimpo
0,"ST. GEORGE — Kaitlyn Larson, a first-year teac...",positive,"[1, 0, 0]",Business,st george kaitlyn larson a firstyear teacher ...
1,"Harare, Zimbabwe – Local businesses are grappl...",neutral,"[0, 1, 0]",Business,harare zimbabwe – local businesses are grappli...
2,(marketscreener.com) Billionaire Elon Musk has...,positive,"[1, 0, 0]",Business,marketscreenercom billionaire elon musk has do...
3,(marketscreener.com) A U.S. trade regulator on...,negative,"[0, 0, 1]",Business,marketscreenercom a us trade regulator on frid...
4,4.5 million households in the U.S. have solar ...,positive,"[1, 0, 0]",Business,45 million households in the us have solar pan...
...,...,...,...,...,...
3495,QRG Capital Management Inc. increased its stak...,positive,"[1, 0, 0]",Technology,qrg capital management inc increased its stake...
3496,QRG Capital Management Inc. bought a new posit...,positive,"[1, 0, 0]",Technology,qrg capital management inc bought a new positi...
3497,QRG Capital Management Inc. boosted its stake ...,positive,"[1, 0, 0]",Technology,qrg capital management inc boosted its stake i...
3498,"WESTFORD, Mass., July 18, 2024 /PRNewswire/ --...",neutral,"[0, 1, 0]",Technology,westford mass july 18 2024 prnewswire accordin...


In [29]:
df["Type"].unique()

<StringArray>
[     'Business', 'Entertainment',       'General',        'Health',
       'Science',        'Sports',    'Technology']
Length: 7, dtype: str

In [30]:
MAX_PALAVRAS = 3000

In [31]:
vetorizador = CountVectorizer(max_features = MAX_PALAVRAS)

In [32]:
texto_vetorizado = vetorizador.fit_transform( df["TextoLimpo"] )
texto_vetorizado

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 98285 stored elements and shape (3500, 3000)>

In [33]:
X = torch.tensor( texto_vetorizado.toarray(), dtype=torch.float32)
X

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [35]:
Y = torch.tensor(df["sentimento_one_hot"], dtype=torch.float32)
Y

tensor([[1., 0., 0.],
        [0., 1., 0.],
        [1., 0., 0.],
        ...,
        [1., 0., 0.],
        [0., 1., 0.],
        [1., 0., 0.]])

In [36]:
vocabulario = vetorizador.get_feature_names_out()

In [37]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.20, random_state = 100)

In [38]:
modelo = nn.Sequential( 
    nn.Linear(in_features=MAX_PALAVRAS, out_features=3),
    # nn.Linear(in_features=MAX_PALAVRAS, out_features=512),
    # nn.Linear(in_features=512, out_features=1),
    # nn.Sigmoid()
)
modelo

Sequential(
  (0): Linear(in_features=3000, out_features=3, bias=True)
)

In [39]:
for param in modelo.parameters():
    print(param)

Parameter containing:
tensor([[-0.0017, -0.0127,  0.0133,  ..., -0.0180,  0.0085, -0.0180],
        [ 0.0164, -0.0174, -0.0132,  ..., -0.0041, -0.0122,  0.0047],
        [ 0.0060,  0.0074, -0.0087,  ..., -0.0013,  0.0121, -0.0080]],
       requires_grad=True)
Parameter containing:
tensor([-0.0053, -0.0078,  0.0098], requires_grad=True)


In [40]:
funcao_perda = nn.CrossEntropyLoss()

In [41]:
otimizador = optim.SGD( modelo.parameters(), lr = 0.1)

In [43]:
epocas = 500
for epoca in range(epocas):
    y_hat = modelo( X_train )
    loss = funcao_perda( y_hat, Y_train )
    if epoca % 100 == 0: 
        print("Loss: ", loss)
    loss.backward()
    otimizador.step()

Loss:  tensor(6.9939e-07, grad_fn=<DivBackward1>)
Loss:  tensor(1.0892e-06, grad_fn=<DivBackward1>)
Loss:  tensor(1.7758e-06, grad_fn=<DivBackward1>)
Loss:  tensor(2.6098e-06, grad_fn=<DivBackward1>)
Loss:  tensor(3.1412e-06, grad_fn=<DivBackward1>)


In [24]:
# acertos = 0 

In [25]:
# texto = df["TextoLimpo"].loc[3496]


In [26]:
# teste_vetorizador = CountVectorizer(vocabulary = vocabulario)

In [27]:
# X_test = teste_vetorizador.fit_transform( df["TextoLimpo"] )
# X_test.toarray()

In [28]:
# teste_vetorizador.get_feature_names_out()

In [29]:
# X_test = torch.tensor( X_test.toarray(), dtype=torch.float32 ) 
# X_test.shape

In [44]:
y_hat_test = modelo( X_test )
y_hat_test

tensor([[-216.9860,  406.9200, -190.0058],
        [ -37.4787,  292.7585, -255.3878],
        [ 966.7238, -312.6717, -654.2052],
        ...,
        [ 151.8871, -184.6499,   32.7825],
        [ 233.4522, -119.4969, -114.0457],
        [ 310.7687, -197.0166, -113.6648]], grad_fn=<AddmmBackward0>)

In [52]:
acertos = 0
for idx, y_hat in enumerate(y_hat_test):
    y = Y_test[idx]
    y_index = torch.argmax( y )
    y_hat_index = torch.argmax( y_hat )
    # resultado_oficial = Y_test[idx].item()
    # resultado_previsto = 0
    # if y_hat.item() >= 0.5:
    #     resultado_previsto = 1
    if y_index == y_hat_index:
        acertos = acertos + 1
    print(f"{y_hat}=>{y_hat_index} | {y} => {y_index}")
print("Porcentagem de acerto: ",  (acertos / len(y_hat_test)) * 100 )    

tensor([-216.9860,  406.9200, -190.0058], grad_fn=<UnbindBackward0>)=>1 | tensor([0., 1., 0.]) => 1
tensor([ -37.4787,  292.7585, -255.3878], grad_fn=<UnbindBackward0>)=>1 | tensor([0., 1., 0.]) => 1
tensor([ 966.7238, -312.6717, -654.2052], grad_fn=<UnbindBackward0>)=>0 | tensor([1., 0., 0.]) => 0
tensor([ 519.2092, -702.2241,  183.1143], grad_fn=<UnbindBackward0>)=>0 | tensor([0., 0., 1.]) => 2
tensor([ 636.1210, -227.5637, -408.7502], grad_fn=<UnbindBackward0>)=>0 | tensor([1., 0., 0.]) => 0
tensor([ 239.0228, -221.9166,  -16.8874], grad_fn=<UnbindBackward0>)=>0 | tensor([1., 0., 0.]) => 0
tensor([-272.2610,  671.8472, -399.9415], grad_fn=<UnbindBackward0>)=>1 | tensor([0., 1., 0.]) => 1
tensor([ 309.6625, -127.5140, -182.2730], grad_fn=<UnbindBackward0>)=>0 | tensor([1., 0., 0.]) => 0
tensor([ 868.5778, -622.9871, -245.8302], grad_fn=<UnbindBackward0>)=>0 | tensor([1., 0., 0.]) => 0
tensor([ 1172.3693, -1056.6765,  -115.8165], grad_fn=<UnbindBackward0>)=>0 | tensor([1., 0., 0.]) =>